# LC 81 — Search in Rotated Sorted Array II
**Day-78 | Binary Search on Rotated Arrays | Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">

**Core Insight:** Duplicates break the guarantee that we can tell
which half is sorted by comparing `nums[lo]` and `nums[mid]`.
The fix is surgical: when `nums[lo] == nums[mid] == nums[hi]`,
just shrink both ends by one (`lo++; hi--`) and continue.
This collapses to O(n) worst-case but stays O(log n) on average.

</div>

## Official Problem Statement

Given an integer array `nums` sorted in ascending order and possibly
rotated at an unknown pivot, and an integer `target`, return `True`
if `target` is in `nums`, or `False` otherwise.

**This version allows duplicate values.**

**Constraints:**
- `1 <= nums.length <= 5000`
- `-10^4 <= nums[i], target <= 10^4`
- `nums` is an ascending array, possibly rotated, with duplicates.

## What This Is Actually Asking

This is LC 33 with one added twist: **values may repeat**.  
That one change creates a dangerous ambiguity:

```
nums = [1, 1, 1, 3, 1]    lo=0, hi=4, mid=2
nums[lo]=1, nums[mid]=1  →  Is left half sorted or not?
```

We genuinely **cannot tell** — `[1,1,1,3]` is sorted, but so is
`[1,3,1,1]` from the right side. When both endpoints equal mid,
no information is gained. The only safe move: shrink both bounds
by one and repeat. This is O(n) in the absolute worst case
(all identical values), but that case is rare in practice.

## Walk Through an Example by Hand

```
nums   = [2, 5, 6, 0, 0, 1, 2]   target = 0
indices:  0  1  2  3  4  5  6
```

**Step 1:** lo=0, hi=6, mid=3  
- nums[mid]=0 == target → **return True** ✓  

---

```
nums   = [2, 5, 6, 0, 0, 1, 2]   target = 3
```

**Step 1:** lo=0, hi=6, mid=3 → nums[mid]=0 ≠ 3  
- nums[lo]=2 > nums[mid]=0 → **right half sorted**  
- 3 in (0..2]? No → hi=2

**Step 2:** lo=0, hi=2, mid=1 → nums[mid]=5 ≠ 3  
- nums[lo]=2 <= nums[mid]=5 → **left half sorted**  
- 3 in [2..5)? Yes → hi=0

**Step 3:** lo=0, hi=0, mid=0 → nums[0]=2 ≠ 3  
- nums[lo]=2 <= nums[mid]=2 → left sorted  
- 3 in [2..2)? No → lo=1 > hi → **return False** ✓

---

**Ambiguous case:**

```
nums   = [1, 1, 1, 3, 1]   target = 3
```

**Step 1:** lo=0, hi=4, mid=2  
- nums[lo]=1 == nums[mid]=1 == nums[hi]=1 → **shrink both**  
- lo=1, hi=3

**Step 2:** lo=1, hi=3, mid=2 → nums[mid]=1 ≠ 3  
- nums[lo]=1 <= nums[mid]=1 → left sorted  
- 3 in [1..1)? No → lo=3

**Step 3:** lo=3, hi=3, mid=3 → nums[3]=3 == target → **True** ✓

## The Picture

```
Normal case (LC 33 logic applies):

 idx:  0    1    2    3    4    5    6
      [2,   5,   6,   0,   0,   1,   2]
       ^              ^              ^
       lo            mid            hi

  nums[lo]=2 > nums[mid]=0  →  RIGHT side is sorted

  ┌─ contains pivot ──────┐   ┌─ sorted ──────┐
  [ 2    5    6    0   |   0    1    2 ]
    lo           mid-1     mid        hi

  If target ∈ (nums[mid], nums[hi]]  →  lo = mid + 1
  Else                               →  hi = mid - 1

---
Ambiguous / duplicate case (new in LC 81):

 idx:  0    1    2    3    4
      [1,   1,   1,   3,   1]
       ^         ^         ^
       lo       mid        hi

  nums[lo]=1 == nums[mid]=1 == nums[hi]=1
  Can't tell which side is sorted!

  SOLUTION: shrink both ends → lo++, hi--

 idx:  1    2    3
      [1,   1,   3]
       ^    ^    ^
       lo  mid   hi

  Now nums[lo]=1 <= nums[mid]=1 → left sorted
  target=3 not in [1,1) → lo = mid+1 = 3
  lo==hi==3, nums[3]=3 → FOUND
```

## When To Use This Pattern

Use **LC 81 logic** when:

- The array is sorted, possibly rotated, and has **duplicates**.
- You need a boolean result (exists or not).
- O(log n) average is acceptable; worst-case O(n) is tolerable.

**LC 33 vs LC 81:**

| Feature           | LC 33       | LC 81          |
|-------------------|-------------|----------------|
| Duplicates        | No          | Yes            |
| Returns           | Index       | Boolean        |
| Worst-case time   | O(log n)    | O(n)           |
| Extra logic       | None        | `lo++; hi--`   |

**Do NOT use when:**
- You need the exact index (LC 33 is more appropriate if unique).
- The array is completely unsorted.

## The Approach

```
LC 33 skeleton + one early-exit clause for duplicates:

while lo <= hi:
    mid = (lo + hi) // 2
    if nums[mid] == target: return True

    # NEW: ambiguous — can't determine sorted side
    if nums[lo] == nums[mid] == nums[hi]:
        lo += 1
        hi -= 1
        continue

    # Same as LC 33 from here
    if nums[lo] <= nums[mid]:          # left half sorted
        if nums[lo] <= target < nums[mid]:
            hi = mid - 1
        else:
            lo = mid + 1
    else:                              # right half sorted
        if nums[mid] < target <= nums[hi]:
            lo = mid + 1
        else:
            hi = mid - 1

return False
```

**Why `lo++; hi--` is safe:** If `nums[lo]==nums[mid]==nums[hi]`,
then `nums[lo]` and `nums[hi]` are not the target (we already
checked `nums[mid] != target`), so shrinking is lossless.

In [ ]:
from typing import List

In [ ]:
def test_harness(func):
    """Run test cases and print PASSED / FAILED summary."""
    cases = [
        # (nums, target, expected)
        ([2, 5, 6, 0, 0, 1, 2], 0,  True),
        ([2, 5, 6, 0, 0, 1, 2], 3,  False),
        ([1, 1, 1, 3, 1],       3,  True),
        ([1, 1, 1, 3, 1],       2,  False),
        ([1],                   1,  True),
        ([1],                   0,  False),
        ([1, 3, 1, 1, 1],       3,  True),
        ([3, 1, 1],             3,  True),
        ([1, 1, 3],             3,  True),   # no rotation
        ([2, 2, 2, 2, 2],       2,  True),   # all same
    ]
    passed = 0
    for nums, target, expected in cases:
        result = func(nums, target)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status}: nums={nums} target={target} "
                f"expected={expected} got={result}"
            )
    total = len(cases)
    print(f"\nResult: {passed}/{total} tests passed.")

In [ ]:
def search(nums: List[int], target: int) -> bool:
    """
    LC 81 — Search in Rotated Sorted Array II.

    A sorted array with duplicates is rotated at an unknown pivot.
    Return True if target exists, False otherwise.

    Strategy:
    ---------
    Same as LC 33, but add a guard before the sorted-half check:
    if nums[lo] == nums[mid] == nums[hi], we cannot determine
    which half is sorted, so safely shrink both ends by one.
    Then apply the standard LC 33 two-half logic.

    Args:
        nums:   Rotated sorted array, may contain duplicates.
        target: Integer value to search for.

    Returns:
        True if target is in nums, False otherwise.

    Examples:
        >>> search([2,5,6,0,0,1,2], 0)
        True
        >>> search([2,5,6,0,0,1,2], 3)
        False
        >>> search([1,1,1,3,1], 3)
        True
    """
    # Debug: visualise initial state
    print(f"[debug] nums={nums}  target={target}")

    pass

    # After implementing, add per-step debug inside the loop:
    # print(f"  lo={lo} hi={hi} mid={mid} "
    #       f"nums[mid]={nums[mid]}")

In [ ]:
# Uncomment and run when solution is ready
# test_harness(search)

## Complexity

| Dimension | Value         | Reason |
|-----------|---------------|--------|
| Time  | O(log n) avg  | Each step halves the search space |
| Time  | O(n) worst    | All identical values, e.g. `[1,1,1,1]` |
| Space | O(1)          | Only pointer variables used |

**When does worst-case occur?**  
When `nums[lo] == nums[mid] == nums[hi]` at every iteration — each
step only trims one element from each side, reducing by 2 instead
of halving. LeetCode judges accept this because the duplicate
scenario is unavoidable without additional structure.

## Real World Connection

**Event log deduplication search:**  
A distributed system writes timestamped events to a ring buffer.
Due to clock skew and retry logic, consecutive entries may carry
identical timestamps. When you need to locate a specific event ID
within a time window, the LC 81 pattern applies: the buffer is
mostly sorted (rotated at the write pointer), but tie timestamps
create the ambiguous-side problem. The `lo++; hi--` shrink handles
ties cleanly and the search degrades gracefully to linear only in
the pathological all-same-timestamp case.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra